In [5]:
# Import libraries for data handling
import pandas as pd
import numpy as np

# Import machine learning utilities from scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Import models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# Import evaluation metrics
from sklearn.metrics import accuracy_score, f1_score


# Load the dataset
data = pd.read_csv("loan_data.csv")

# Display first 5 rows to understand dataset structure
print(data.head())



# Feature Engineering

# Create a new feature that represents how large the loan is relative to income
# Higher ratio may indicate higher risk
data["LoanIncomeRatio"] = data["LoanAmount"] / data["ApplicantIncome"]

# Replace infinite values (can happen if income = 0)
data["LoanIncomeRatio"] = data["LoanIncomeRatio"].replace([np.inf, -np.inf], np.nan)

# Replace missing values with 0
data["LoanIncomeRatio"] = data["LoanIncomeRatio"].fillna(0)


# Encode Categorical Variables

# Machine learning models cannot process text categories
# LabelEncoder converts them into numeric values

le_education = LabelEncoder()
le_married = LabelEncoder()
le_loan_status = LabelEncoder()

# Convert Education (Graduate / Not Graduate) into numbers
data["Education"] = le_education.fit_transform(data["Education"])

# Convert Married (Yes / No) into numbers
data["Married"] = le_married.fit_transform(data["Married"])

# Convert target variable Loan_Status (Y / N) into numbers
data["Loan_Status"] = le_loan_status.fit_transform(data["Loan_Status"])



# Define Features (X) and Target (y)


# X contains the input features used to predict loan approval
X = data[[
    "ApplicantIncome",
    "LoanAmount",
    "Credit_History",
    "Education",
    "Married",
    "LoanIncomeRatio"
]]

# y is the target variable we want to predict
y = data["Loan_Status"]



# Split Dataset into Training and Testing Sets

# 80% data used for training
# 20% used for testing model performance

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# Train Decision Tree Model
dt = DecisionTreeClassifier()

# Train model using training data
dt.fit(X_train, y_train)

# Make predictions on test data
dt_pred = dt.predict(X_test)


# Train Random Forest Model
# Random Forest builds multiple decision trees and averages results

rf = RandomForestClassifier(n_estimators=100)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)


# Train Gradient Boosting Model

# HistGradientBoostingClassifier is an optimized gradient boosting model
# It also supports missing values (NaN)

gb = HistGradientBoostingClassifier()

gb.fit(X_train, y_train)

gb_pred = gb.predict(X_test)


# Evaluation Function
# This function calculates and prints model performance

def evaluate(y_test, pred, model_name):

    # Accuracy = percentage of correct predictions
    acc = accuracy_score(y_test, pred)

    # F1 Score balances precision and recall
    f1 = f1_score(y_test, pred)

    print(model_name)
    print("Accuracy:", acc)
    print("F1 Score:", f1)
    print()


# Evaluate All Models

evaluate(y_test, dt_pred, "Decision Tree")

evaluate(y_test, rf_pred, "Random Forest")

evaluate(y_test, gb_pred, "Gradient Boosting")

    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Graduate            No   
1  LP001003   Male     Yes          1      Graduate            No   
2  LP001005   Male     Yes          0      Graduate           Yes   
3  LP001006   Male     Yes          0  Not Graduate            No   
4  LP001008   Male      No          0      Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0         NaN             360.0   
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0             360.0   
3             2583             2358.0       120.0             360.0   
4             6000                0.0       141.0             360.0   

   Credit_History Property_Area Loan_Status  
0             1.0         Urban           Y  
1             1.0         Rural           N  
2             1.0   

In [8]:
print("\n====== Loan Approval Prediction System ======\n")

# User inputs
applicant_income = float(input("Enter Applicant Income: "))
loan_amount = float(input("Enter Loan Amount: "))
credit_history = int(input("Credit History (1 = Good, 0 = Bad): "))

education = int(input("Education (1 = Graduate, 0 = Not Graduate): "))
married = int(input("Married (1 = Yes, 0 = No): "))


# Feature engineering for user input
loan_income_ratio = loan_amount / applicant_income if applicant_income != 0 else 0


# Create dataframe for prediction
user_data = pd.DataFrame([[
    applicant_income,
    loan_amount,
    credit_history,
    education,
    married,
    loan_income_ratio
]], columns=X.columns)


# Predict using Random Forest (best model usually)
prediction = rf.predict(user_data)


# Display result

if prediction[0] == 1:
    print("\nLoan Approved")
else:
    print("\nLoan Rejected")


====== Loan Approval Prediction System ======

Enter Applicant Income: 200000
Enter Loan Amount: 1500000
Credit History (1 = Good, 0 = Bad): 1
Education (1 = Graduate, 0 = Not Graduate): 1
Married (1 = Yes, 0 = No): 1

Loan Approved
